# 🌿 PlantHealth AI — Transfer Learning with ResNet50

## Notebook 02: High-Performance Model via Transfer Learning

In this notebook, we use **Transfer Learning** with a pre-trained **ResNet50** backbone to achieve high accuracy on the PlantVillage dataset. Transfer Learning leverages knowledge from ImageNet (1.2M images, 1000 classes) and adapts it for our plant disease task.

### Outline
1. Why Transfer Learning?
2. Model Architecture — ResNet50 with Custom Head
3. Stage 1: Frozen Backbone Training
4. Stage 2: Fine-Tuning
5. Combined Training History
6. Comprehensive Evaluation
7. Confusion Matrix & Error Analysis
8. Comparison with Custom CNN
9. Model Export

---

## 1. Setup

In [ ]:
import sys
import os
import json
import time

# Add project root to path
PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), '..'))
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import yaml
from pathlib import Path
from PIL import Image

# Project modules
from src.data.dataset import PlantDiseaseDataset, get_data_loaders
from src.data.transforms import get_train_transforms, get_val_transforms, denormalize
from src.models.resnet_transfer import PlantDiseaseResNet, create_resnet_model
from src.models.custom_cnn import PlantDiseaseCNN
from src.training.trainer import Trainer, get_optimizer, get_scheduler
from src.evaluation.metrics import evaluate_model, find_misclassified_samples, find_most_confused_classes
from src.utils.visualization import (
    plot_training_history, plot_confusion_matrix,
    plot_sample_predictions, plot_class_distribution
)

# Settings
plt.rcParams['figure.figsize'] = (12, 8)
plt.rcParams['font.size'] = 12
sns.set_style('whitegrid')

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

In [ ]:
# Load config
config_path = Path(PROJECT_ROOT) / 'config.yaml'
with open(config_path, 'r') as f:
    config = yaml.safe_load(f)

# Paths
TRAIN_DIR = Path(PROJECT_ROOT) / config['paths']['train_dir']
VALID_DIR = Path(PROJECT_ROOT) / config['paths']['valid_dir']
TEST_DIR = Path(PROJECT_ROOT) / 'data' / 'New Plant Diseases Dataset' / 'test'
CHECKPOINT_DIR = Path(PROJECT_ROOT) / config['paths']['checkpoint_dir']
REPORTS_DIR = Path(PROJECT_ROOT) / 'reports' / 'figures'
REPORTS_DIR.mkdir(parents=True, exist_ok=True)

print(f"Train dir: {TRAIN_DIR} (exists: {TRAIN_DIR.exists()})")
print(f"Valid dir: {VALID_DIR} (exists: {VALID_DIR.exists()})")
print(f"Test dir:  {TEST_DIR} (exists: {TEST_DIR.exists()})")

## 2. Why Transfer Learning?

| Aspect | Custom CNN | Transfer Learning (ResNet50) |
|--------|-----------|-----------------------------|
| **Parameters** | ~600K | ~25M (23M frozen initially) |
| **Training data needed** | Lots | Less (pre-trained features) |
| **Training time** | Long | Short (Stage 1: minutes) |
| **Expected accuracy** | 70-85% | 90-98% |
| **Feature quality** | Learned from scratch | ImageNet-quality features |

In [ ]:
# Compare parameter counts
num_classes = config['model']['num_classes']

# Custom CNN
custom_model = PlantDiseaseCNN(num_classes=num_classes)
custom_trainable, custom_total = custom_model.count_parameters()

# ResNet50 (frozen)
resnet_frozen = PlantDiseaseResNet(num_classes=num_classes, pretrained=False, freeze_backbone=True)
frozen_trainable, frozen_total = resnet_frozen.count_parameters()

# ResNet50 (unfrozen)
resnet_unfrozen = PlantDiseaseResNet(num_classes=num_classes, pretrained=False, freeze_backbone=False)
unfrozen_trainable, unfrozen_total = resnet_unfrozen.count_parameters()

print(f"{'Model':<35} {'Trainable':>15} {'Total':>15}")
print("-" * 65)
print(f"{'Custom CNN':<35} {custom_trainable:>15,} {custom_total:>15,}")
print(f"{'ResNet50 (backbone frozen)':<35} {frozen_trainable:>15,} {frozen_total:>15,}")
print(f"{'ResNet50 (fully unfrozen)':<35} {unfrozen_trainable:>15,} {unfrozen_total:>15,}")

# Cleanup
del custom_model, resnet_frozen, resnet_unfrozen

## 3. Model Architecture

```
ResNet50 Backbone (pretrained on ImageNet)
  → AdaptiveAvgPool2d(1, 1)  → Flatten  → (2048,)
  → Linear(2048 → 512) + ReLU + Dropout(0.5)
  → Linear(512 → 38)
```

**Two-Stage Training:**
1. **Stage 1** (Frozen): Only train the classifier head → fast convergence
2. **Stage 2** (Fine-tune): Unfreeze last 2 layer groups + lower LR → squeeze out extra accuracy

In [ ]:
# ============================================================
# TRAINING TOGGLE
# Set TRAIN_FROM_SCRATCH = True to train the model from scratch
# Set TRAIN_FROM_SCRATCH = False to load a pre-trained checkpoint
# ============================================================
TRAIN_FROM_SCRATCH = False

CHECKPOINT_PATH = CHECKPOINT_DIR / 'resnet50_best.pth'

In [ ]:
# Create data loaders
train_loader, val_loader, class_to_idx, class_names = get_data_loaders(
    train_dir=str(TRAIN_DIR),
    valid_dir=str(VALID_DIR),
    batch_size=config['training']['batch_size'],
    num_workers=config['training']['num_workers'],
    image_size=config['model']['image_size'],
    pin_memory=config['training']['pin_memory'],
    augmentation_config=config.get('augmentation', {}).get('train', {}),
)

print(f"Train batches: {len(train_loader)}")
print(f"Val batches:   {len(val_loader)}")
print(f"Num classes:   {len(class_names)}")

In [ ]:
if TRAIN_FROM_SCRATCH:
    # ============================================
    # STAGE 1: Frozen backbone (train head only)
    # ============================================
    print("=" * 60)
    print("STAGE 1: Training classifier head (backbone frozen)")
    print("=" * 60)
    
    tl_config = config['training']['transfer_learning']
    
    # Create model with frozen backbone
    model = create_resnet_model(
        model_name=config['model']['pretrained_model'],
        num_classes=num_classes,
        dropout_rate=config['regularization']['dropout'],
        pretrained=True,
        freeze_backbone=True,
        device=device,
    )
    
    criterion = nn.CrossEntropyLoss(label_smoothing=config['regularization']['label_smoothing'])
    
    # Only optimize classifier parameters
    optimizer_s1 = get_optimizer(
        model,
        optimizer_name=config['optimizer']['name'],
        learning_rate=tl_config['stage1']['learning_rate'],
    )
    
    scheduler_s1 = get_scheduler(
        optimizer_s1,
        scheduler_name=config['scheduler']['name'],
        factor=config['scheduler']['factor'],
        patience=config['scheduler']['patience'],
        min_lr=config['scheduler']['min_lr'],
    )
    
    trainer_s1 = Trainer(
        model=model,
        train_loader=train_loader,
        val_loader=val_loader,
        criterion=criterion,
        optimizer=optimizer_s1,
        scheduler=scheduler_s1,
        device=device,
        checkpoint_dir=str(CHECKPOINT_DIR),
        use_amp=torch.cuda.is_available(),
    )
    
    history_s1 = trainer_s1.train(
        epochs=tl_config['stage1']['epochs'],
        early_stopping_patience=10,
        checkpoint_metric='val_accuracy',
    )
    
    # Save stage 1 checkpoint
    trainer_s1.save_model(str(CHECKPOINT_DIR / 'resnet50_stage1.pth'))
    print("\nStage 1 complete!")
    
    # ============================================
    # STAGE 2: Fine-tune (unfreeze last layers)
    # ============================================
    print("\n" + "=" * 60)
    print("STAGE 2: Fine-tuning (unfreezing last layers)")
    print("=" * 60)
    
    # Unfreeze last N layer groups
    model.unfreeze_last_n_layers(tl_config['stage2']['unfreeze_layers'])
    
    trainable, total = model.count_parameters()
    print(f"Trainable params after unfreeze: {trainable:,} / {total:,}")
    
    optimizer_s2 = get_optimizer(
        model,
        optimizer_name=config['optimizer']['name'],
        learning_rate=tl_config['stage2']['learning_rate'],
    )
    
    scheduler_s2 = get_scheduler(
        optimizer_s2,
        scheduler_name=config['scheduler']['name'],
        factor=config['scheduler']['factor'],
        patience=config['scheduler']['patience'],
        min_lr=config['scheduler']['min_lr'],
    )
    
    trainer_s2 = Trainer(
        model=model,
        train_loader=train_loader,
        val_loader=val_loader,
        criterion=criterion,
        optimizer=optimizer_s2,
        scheduler=scheduler_s2,
        device=device,
        checkpoint_dir=str(CHECKPOINT_DIR),
        use_amp=torch.cuda.is_available(),
    )
    
    history_s2 = trainer_s2.train(
        epochs=tl_config['stage2']['epochs'],
        early_stopping_patience=config['callbacks']['early_stopping']['patience'],
        checkpoint_metric='val_accuracy',
    )
    
    # Combine histories
    history = {
        'train_loss': history_s1['train_loss'] + history_s2['train_loss'],
        'train_acc': history_s1['train_acc'] + history_s2['train_acc'],
        'val_loss': history_s1['val_loss'] + history_s2['val_loss'],
        'val_acc': history_s1['val_acc'] + history_s2['val_acc'],
        'lr': history_s1['lr'] + history_s2['lr'],
    }
    
    # Save final model
    trainer_s2.save_model(str(CHECKPOINT_DIR / 'resnet50_best.pth'))
    print("\nStage 2 complete! Model saved.")

else:
    print(f"Loading pre-trained checkpoint: {CHECKPOINT_PATH}")
    
    # Create model architecture (no pretrained ImageNet weights needed)
    model = create_resnet_model(
        model_name=config['model']['pretrained_model'],
        num_classes=num_classes,
        dropout_rate=config['regularization']['dropout'],
        pretrained=False,
        freeze_backbone=False,
        device=device,
    )
    
    if CHECKPOINT_PATH.exists():
        checkpoint = torch.load(CHECKPOINT_PATH, map_location=device, weights_only=False)
        
        if 'model_state_dict' in checkpoint:
            model.load_state_dict(checkpoint['model_state_dict'])
            history = checkpoint.get('history', {})
        else:
            model.load_state_dict(checkpoint)
            history = {}
        
        model.to(device)
        model.eval()
        print("Checkpoint loaded successfully!")
        
        if history:
            print(f"Training epochs recorded: {len(history.get('train_loss', []))}")
            if 'val_acc' in history:
                print(f"Best validation accuracy: {max(history['val_acc']):.2f}%")
    else:
        print(f"WARNING: Checkpoint not found at {CHECKPOINT_PATH}")
        print("Set TRAIN_FROM_SCRATCH = True to train the model")
        history = {}

## 5. Training History Visualization

In [ ]:
if history and 'train_loss' in history and len(history['train_loss']) > 0:
    fig = plot_training_history(
        history,
        title='ResNet50 Transfer Learning — Training History (Both Stages)',
        save_path=str(REPORTS_DIR / 'resnet50_training_curves.png')
    )
    
    # Add vertical line for stage boundary if we know it
    tl_config = config['training']['transfer_learning']
    stage1_epochs = tl_config['stage1']['epochs']
    if len(history['train_loss']) > stage1_epochs:
        for ax in fig.axes:
            ax.axvline(x=stage1_epochs, color='purple', linestyle='--', alpha=0.6, label='Stage 1→2')
            ax.legend()
    
    plt.show()
else:
    print("No training history available. Train from scratch to see training curves.")

## 6. Comprehensive Evaluation on Test Set

In [ ]:
from torch.utils.data import DataLoader

val_transforms = get_val_transforms(image_size=config['model']['image_size'])

test_dataset = PlantDiseaseDataset(
    str(TEST_DIR),
    transform=val_transforms,
    class_to_idx=class_to_idx,
)

test_loader = DataLoader(
    test_dataset,
    batch_size=config['training']['batch_size'],
    shuffle=False,
    num_workers=config['training']['num_workers'],
    pin_memory=True,
)

print(f"Test samples: {len(test_dataset):,}")
print(f"Test batches: {len(test_loader)}")

In [ ]:
# Run comprehensive evaluation
model.eval()
eval_results = evaluate_model(
    model=model,
    dataloader=test_loader,
    device=device,
    class_names=class_names,
    print_report=True,
)

In [ ]:
# Summary metrics
print("\n" + "="*50)
print("RESNET50 TRANSFER LEARNING — SUMMARY METRICS")
print("="*50)
print(f"Overall Accuracy:     {eval_results['accuracy']:.2f}%")
print(f"Top-5 Accuracy:       {eval_results['top5_accuracy']:.2f}%")
print(f"Macro Precision:      {eval_results['macro_precision']:.4f}")
print(f"Macro Recall:         {eval_results['macro_recall']:.4f}")
print(f"Macro F1-Score:       {eval_results['macro_f1']:.4f}")
print(f"Weighted F1-Score:    {eval_results['weighted_f1']:.4f}")

## 7. Confusion Matrix & Error Analysis

In [ ]:
# Confusion matrix
fig = plot_confusion_matrix(
    eval_results['confusion_matrix'],
    class_names=class_names,
    title='ResNet50 — Confusion Matrix (Normalized)',
    normalize=True,
    save_path=str(REPORTS_DIR / 'resnet50_confusion_matrix.png'),
)
plt.show()

In [ ]:
# Most confused class pairs
confused_pairs = find_most_confused_classes(
    eval_results['confusion_matrix'],
    class_names,
    top_n=10,
)

print("\nTop 10 Most Confused Class Pairs:")
print("-" * 70)
for i, pair in enumerate(confused_pairs, 1):
    print(f"{i:2d}. True: {pair['true_class'][:35]:35s}  →  Pred: {pair['predicted_class'][:35]:35s}  ({pair['count']} times)")

In [ ]:
# Most confident misclassifications
misclassified = find_misclassified_samples(
    eval_results['predictions'],
    eval_results['true_labels'],
    eval_results['probabilities'],
    n_samples=10,
    class_names=class_names,
)

print("\nTop 10 Most Confident Misclassifications:")
print("-" * 80)
for i, m in enumerate(misclassified, 1):
    print(f"{i:2d}. True: {m['true'][:30]:30s}  Pred: {m['predicted'][:30]:30s}  Conf: {m['confidence']:.1f}%")

In [ ]:
# Sample predictions grid
np.random.seed(42)
indices = np.random.choice(len(test_dataset), size=10, replace=False)

images_list = []
true_labels_list = []
pred_labels_list = []
confidence_list = []

model.eval()
with torch.no_grad():
    for idx in indices:
        image, label = test_dataset[idx]
        output = model(image.unsqueeze(0).to(device))
        probs = torch.softmax(output, dim=1)
        pred_idx = probs.argmax(dim=1).item()
        confidence = probs.max().item() * 100
        
        img_display = denormalize(image.numpy())
        
        images_list.append(img_display)
        true_labels_list.append(class_names[label])
        pred_labels_list.append(class_names[pred_idx])
        confidence_list.append(confidence)

fig = plot_sample_predictions(
    images=images_list,
    true_labels=true_labels_list,
    pred_labels=pred_labels_list,
    confidences=confidence_list,
    title='ResNet50 — Sample Test Predictions',
    figsize=(18, 8),
    save_path=str(REPORTS_DIR / 'resnet50_sample_predictions.png'),
)
plt.show()

correct = sum(1 for t, p in zip(true_labels_list, pred_labels_list) if t == p)
print(f"\nCorrect: {correct}/10")

## 8. Comparison: Custom CNN vs. ResNet50

In [ ]:
# Load Custom CNN results if available
custom_cnn_path = Path(PROJECT_ROOT) / 'reports' / 'custom_cnn_results.json'

if custom_cnn_path.exists():
    with open(custom_cnn_path, 'r') as f:
        custom_results = json.load(f)
    
    resnet_results = {
        'model': 'ResNet50 (Transfer Learning)',
        'accuracy': eval_results['accuracy'],
        'top5_accuracy': eval_results['top5_accuracy'],
        'macro_precision': float(eval_results['macro_precision']),
        'macro_recall': float(eval_results['macro_recall']),
        'macro_f1': float(eval_results['macro_f1']),
        'weighted_f1': float(eval_results['weighted_f1']),
    }
    
    # Comparison table
    print("\n" + "=" * 65)
    print("MODEL COMPARISON: Custom CNN vs. ResNet50")
    print("=" * 65)
    print(f"{'Metric':<25} {'Custom CNN':>15} {'ResNet50':>15} {'Δ':>10}")
    print("-" * 65)
    
    for metric in ['accuracy', 'top5_accuracy', 'macro_precision', 'macro_recall', 'macro_f1', 'weighted_f1']:
        custom_val = custom_results[metric]
        resnet_val = resnet_results[metric]
        delta = resnet_val - custom_val
        sign = '+' if delta >= 0 else ''
        print(f"{metric:<25} {custom_val:>15.2f} {resnet_val:>15.2f} {sign}{delta:>9.2f}")
    
    # Bar chart comparison
    metrics = ['accuracy', 'macro_f1', 'weighted_f1']
    custom_vals = [custom_results[m] for m in metrics]
    resnet_vals = [resnet_results[m] for m in metrics]
    # Scale F1 scores to percentage for comparison
    custom_vals = [v * 100 if v < 1 else v for v in custom_vals]
    resnet_vals = [v * 100 if v < 1 else v for v in resnet_vals]
    
    x = np.arange(len(metrics))
    width = 0.35
    
    fig, ax = plt.subplots(figsize=(10, 6))
    bars1 = ax.bar(x - width/2, custom_vals, width, label='Custom CNN', color='#3498db', alpha=0.8)
    bars2 = ax.bar(x + width/2, resnet_vals, width, label='ResNet50', color='#2ecc71', alpha=0.8)
    
    ax.set_ylabel('Score (%)', fontsize=12)
    ax.set_title('Model Comparison — Custom CNN vs. ResNet50', fontsize=14, fontweight='bold')
    ax.set_xticks(x)
    ax.set_xticklabels([m.replace('_', ' ').title() for m in metrics], fontsize=11)
    ax.legend(fontsize=11)
    ax.set_ylim(0, 105)
    ax.grid(axis='y', alpha=0.3)
    
    # Add value labels
    for bar in bars1 + bars2:
        h = bar.get_height()
        ax.text(bar.get_x() + bar.get_width()/2., h + 1, f'{h:.1f}%', ha='center', va='bottom', fontsize=9)
    
    plt.tight_layout()
    plt.savefig(str(REPORTS_DIR / 'model_comparison.png'), dpi=150, bbox_inches='tight')
    plt.show()

else:
    print("Custom CNN results not found. Run 01_Custom_CNN.ipynb first for comparison.")

## 9. Model Export

Export the trained model for deployment.

In [ ]:
# Save as .pth (PyTorch native format)
export_dir = Path(PROJECT_ROOT) / 'backend' / 'models'
export_dir.mkdir(parents=True, exist_ok=True)

# Save model state dict
export_path = export_dir / 'plant_disease_model.pth'
torch.save({
    'model_state_dict': model.state_dict(),
    'class_names': class_names,
    'class_to_idx': class_to_idx,
    'num_classes': num_classes,
    'model_architecture': 'resnet50',
    'image_size': config['model']['image_size'],
}, export_path)

print(f"Model exported to: {export_path}")
print(f"File size: {export_path.stat().st_size / (1024*1024):.1f} MB")

# Save class names as JSON
class_names_path = export_dir / 'class_names.json'
with open(class_names_path, 'w') as f:
    json.dump(class_names, f, indent=2)
print(f"Class names saved to: {class_names_path}")

In [ ]:
# Save ResNet50 results for report
resnet50_results = {
    'model': 'ResNet50 (Transfer Learning)',
    'accuracy': eval_results['accuracy'],
    'top5_accuracy': eval_results['top5_accuracy'],
    'macro_precision': float(eval_results['macro_precision']),
    'macro_recall': float(eval_results['macro_recall']),
    'macro_f1': float(eval_results['macro_f1']),
    'weighted_precision': float(eval_results['weighted_precision']),
    'weighted_recall': float(eval_results['weighted_recall']),
    'weighted_f1': float(eval_results['weighted_f1']),
}

results_path = Path(PROJECT_ROOT) / 'reports' / 'resnet50_results.json'
with open(results_path, 'w') as f:
    json.dump(resnet50_results, f, indent=2)

print(f"Results saved to: {results_path}")

---

## Summary

In this notebook, we:
1. ✅ Compared Custom CNN vs Transfer Learning approaches
2. ✅ Built a ResNet50 model with custom classifier head  
3. ✅ Implemented two-stage training (frozen → fine-tuned)
4. ✅ Achieved high accuracy with Transfer Learning
5. ✅ Generated comprehensive evaluation metrics
6. ✅ Analyzed confusion patterns and errors
7. ✅ Compared results with the baseline Custom CNN
8. ✅ Exported the model for deployment

**Next step**: Deploy the model with the Streamlit web app (`streamlit_app.py`).